In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
file_path = dbutils.widgets.get("atb_report_path")
filename_format = dbutils.widgets.get("filename_format")
landing_table = dbutils.widgets.get("landing_table")

In [0]:
%skip
fetch_date="2026-01-17"
file_path ='/Volumes/dev_bronze_landing/ability/alphacollector_files/atb_files/'
filename_format='Bayada_ATB_Report_YYYYMMDD.txt'
landing_table='dev_bronze_landing.alphacollector.agedtrailbalance_report'

In [0]:
file_date = fetch_date[0:4] + fetch_date[5:7] + fetch_date[8:10]
file_name = filename_format.replace("YYYYMMDD", file_date)
file_path = file_path + file_name
print(file_path)

In [0]:
df_atb_report = spark.read.option("header", "true").option("delimiter", "|").csv(file_path)
df_atb_report.createOrReplaceTempView("atb_report_table_view")

In [0]:
display(
spark.sql(f"""
DELETE FROM {landing_table}
WHERE _reporting_date = CAST('{fetch_date}' AS DATE)
""")
)

In [0]:
display(
spark.sql(f"""
INSERT INTO {landing_table}
SELECT
  `Group` AS `Group`,
  ClientName AS ClientName,
  Facility AS Facility,
  FacilityCode AS FacilityCode,
  Patient AS Patient,
  PatientDOB AS PatientDOB,
  PatientType AS PatientType,
  AccountNumber AS AccountNumber,
  MedicalRecordNumber AS MedicalRecordNumber,
  AccountAge AS AccountAge,
  ClaimFromDate AS ClaimFromDate,
  ClaimThruDate AS ClaimThruDate,
  AdmitDate AS AdmitDate,
  DischargeDate AS DischargeDate,
  PlacementDate AS PlacementDate,
  PlacementUpdated AS PlacementUpdated,
  PayerCategory AS PayerCategory,
  ActiveInsName AS ActiveInsName,
  ActiveInsCode AS ActiveInsCode,
  PrimaryInsCode AS PrimaryInsCode,
  PrimaryInsName AS PrimaryInsName,
  PrimaryInsBalance AS PrimaryInsBalance,
  SecondaryInsCode AS SecondaryInsCode,
  SecondaryInsName AS SecondaryInsName,
  SecondaryInsBalance AS SecondaryInsBalance,
  TertiaryInsCode AS TertiaryInsCode,
  TertiaryInsName AS TertiaryInsName,
  TertiaryInsBalance AS TertiaryInsBalance,
  AccountBalance AS AccountBalance,
  PatientBalance AS PatientBalance,
  ExpectedNetRevenue AS ExpectedNetRevenue,
  ClaimStatus AS ClaimStatus,
  Status AS Status,
  AssignedTo AS AssignedTo,
  LastAction AS LastAction,
  DaysUntouched AS DaysUntouched,
  FollowUpDate AS FollowUpDate,
  DelinquentDays AS DelinquentDays,
  LastNote AS LastNote,
  LastNoteBy AS LastNoteBy,
  LastNoteDate AS LastNoteDate,
  PhysicianName AS PhysicianName,
  DRG AS DRG,
  PrimProcCode AS PrimProcCode,
  Agency AS Agency,
  AgencyAssignDate AS AgencyAssignDate,
  ARStatus AS ARStatus,
  LastBillDate AS LastBillDate,
  CurrentFC AS CurrentFC,
  CurrentFCDescription AS CurrentFCDescription,
  ServiceType AS ServiceType,
  TotalAdjustments AS TotalAdjustments,
  TotalCharges AS TotalCharges,
  TotalPayments AS TotalPayments,
  IsHighPriority AS IsHighPriority,
  IsDelinquent AS IsDelinquent,
  IsClosed AS IsClosed,
  ContactType AS ContactType,
  NumOfTouches AS NumOfTouches,
  RiskAssessment AS RiskAssessment,
  ProjectName AS ProjectName,
  ProjectOwner AS ProjectOwner,
  TrackingNumberInternal AS TrackingNumberInternal,
  TrackingNumberExternal AS TrackingNumberExternal,
  LastProjectNoteDate AS LastProjectNoteDate,
  LastProjectNoteBy AS LastProjectNoteBy,
  LastProjectNote AS LastProjectNote,
  CAST("{fetch_date}" AS DATE) AS _reporting_date,
  CAST("{file_name[:-4]}" AS STRING) AS _file_name,
  current_timestamp() AS _load_timestamp
  FROM atb_report_table_view
""")
)